In [13]:
class Node:
    def __init__(self, position:tuple, parent:tuple, g:int, h:int):
        self.position = position
        self.parent = parent
        self.g = g
        self.h = h

    def f(self):
        return self.g + self.h
    def __eq__(self, other):
        if not isinstance(other, Node):
            return NotImplemented
        return self.position == other.position

    def __hash__(self):
        return hash(self.position)

class A_star:
    def __init__(self, hx, input: tuple, goal: tuple, map: list[list[str]],uniform:bool = False):
        self.hx = hx
        self.input = input
        self.goal = goal
        self.map = map
        self.uniform = uniform
        self.result = []
        self.counter = 0
        self.explored: list[tuple[int, int]] = []
        self.candidates=[]
        self.nodes = {}
        self.__initiationNodes__()

    def __initiationNodes__(self):
            for i in range(len(self.map)):
                for j in range(len(self.map[0])):
                    if self.map[i][j] != 'X':
                        self.nodes[(i,j)] = Node(
                            (i,j), None, float('inf'), self.hx((i,j), self.goal)
                        )
    def neighbor (self,current:Node):
        x,y =current.position
        neighbors = []
        directions = [(-1,0),(1,0),(0,-1),(0,1)]
        for dx, dy in directions:
            pos = (x + dx, y + dy)
        
            if pos in self.nodes:
                neighbors.append(self.nodes[pos])
        
        return neighbors

    def expand_one(self,current:Node):
        for neighbor in self.neighbor(current):
            if self.uniform:
                delta_value = 1
            else:
                value = self.map[neighbor.position[0]][neighbor.position[1]]
                if value in ('S','E'):
                    delta_value = 0
                elif value in('A','B','T1','T2','T3','T4'):
                    delta_value = 1
                else:
                    delta_value = int(value)
            new_weight =current.g + delta_value
            if new_weight < neighbor.g:
                neighbor.g = new_weight
                neighbor.parent = current.position
                if neighbor not in self.candidates:
                    self.candidates.append(neighbor)

    def isGoal(self, position:tuple):
        return position == self.goal

    
    def search(self):
        self.counter = 0
        self.explored.clear()
        if self.input not in self.nodes or self.goal not in self.nodes:
            return False
        start = self.nodes[self.input]
        start.g =0
        start.parent =None
        self.candidates.append(start)
        self.result = []
        while self.candidates:
            self.candidates.sort(key = lambda x:x.f())
            current = self.candidates.pop(0)
            self.counter += 1
            self.explored.append(current.position)
            if self.isGoal(current.position):
                self.result = self.reconstruct_path(current)
                return True
            self.expand_one(current)
        return False

    def reconstruct_path(self, current: Node):
            path = []
    
            while current is not None:
                path.append(current.position)
    
                if current.parent is None:
                    break
    
                current = self.nodes[current.parent]
    
            return path[::-1]
    

In [14]:
start = (0, 0)
goal = (3, 4)

grid = [
    ['S', '.', '.', 'X', '.'],
    ['.', 'X', '.', '.', '.'],
    ['.', 'X', '.', 'X', '.'],
    ['.', '.', '.', 'X', 'E'],
    ['X', 'X', '.', '.', '.']
]

In [15]:
def heuristics(current,goal):
    return abs(current[0] - goal[0]) + abs(current[1] - goal[1])


astar = A_star(heuristics, start, goal, grid,uniform=True)

if astar.search():
    print("Path found:", astar.result)
    print("Nodes expanded:", astar.counter)
    print("Expand processed nodes:", astar.explored)
    print("Path length:", astar.nodes[goal].g)
else:
    print("No path found")


Path found: [(0, 0), (0, 1), (0, 2), (1, 2), (1, 3), (1, 4), (2, 4), (3, 4)]
Nodes expanded: 14
Expand processed nodes: [(0, 0), (1, 0), (0, 1), (2, 0), (0, 2), (3, 0), (1, 2), (3, 1), (2, 2), (1, 3), (3, 2), (1, 4), (2, 4), (3, 4)]
Path length: 7


In [16]:

def naive_heuristics(current,goal):
    return 0
astar_naive = A_star(naive_heuristics, start, goal, grid,uniform=True)
if astar_naive.search():
    print("Naive path found:", astar_naive.result)
    print("Naive nodes expanded:", astar_naive.counter)
    print("Naive expand processed nodes:", astar_naive.explored)
    print("Naive path length:", astar_naive.nodes[goal].g)
else:
    print("No naive path found")

Naive path found: [(0, 0), (0, 1), (0, 2), (1, 2), (1, 3), (1, 4), (2, 4), (3, 4)]
Naive nodes expanded: 17
Naive expand processed nodes: [(0, 0), (1, 0), (0, 1), (2, 0), (0, 2), (3, 0), (1, 2), (3, 1), (2, 2), (1, 3), (3, 2), (1, 4), (4, 2), (0, 4), (2, 4), (4, 3), (3, 4)]
Naive path length: 7


In [17]:
weighted_start = (0, 0)
weighted_goal = (4, 4)
weighted_grid = [
    ['S', 1, 1, 8, 1],
    [1, 8, 3, 1, 1],
    [1, 8, 1, 8, 1],
    [1, 1, 1, 8, 1],
    [8, 8, 1, 1, 'E']
]

goal_row, goal_col = weighted_goal
weighted_goal_neighbors = []
for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
    row, col = goal_row + dr, goal_col + dc
    if (0 <= row < len(weighted_grid) and 0 <= col < len(weighted_grid[0])
            and weighted_grid[row][col] != "X"):
        weighted_goal_neighbors.append((row, col))


def weighted_heuristics(current, goal):
    if current == goal:
        return 0
    return min((heuristics(current, neighbor) for neighbor in weighted_goal_neighbors), default=0)


weighted_astar = A_star(weighted_heuristics, weighted_start, weighted_goal, weighted_grid)

if weighted_astar.search():
    print("Path found:", weighted_astar.result)
    print("Weighted nodes expanded:", weighted_astar.counter)
    print("Weighted expand processed nodes:", weighted_astar.explored)
    print("Path length:", weighted_astar.nodes[weighted_goal].g)
else:
    print("No path found")

Path found: [(0, 0), (1, 0), (2, 0), (3, 0), (3, 1), (3, 2), (4, 2), (4, 3), (4, 4)]
Weighted nodes expanded: 11
Weighted expand processed nodes: [(0, 0), (1, 0), (0, 1), (2, 0), (0, 2), (3, 0), (3, 1), (3, 2), (4, 2), (4, 3), (4, 4)]
Path length: 7


In [18]:
weighted_astar_origin = A_star(heuristics, weighted_start, weighted_goal, weighted_grid)
weighted_astar_naive = A_star(naive_heuristics, weighted_start, weighted_goal, weighted_grid)
if weighted_astar_origin.search():
    print("Path found:", weighted_astar_origin.result)
    print("Weighted nodes expanded:", weighted_astar_origin.counter)
    print("Weighted expand processed nodes:", weighted_astar_origin.explored)
    print("Path length:", weighted_astar_origin.nodes[weighted_goal].g)
else:
    print("No path found")

if weighted_astar_naive.search():
    print("Naive path found:", weighted_astar_naive.result)
    print("Naive nodes expanded:", weighted_astar_naive.counter)
    print("Naive expand processed nodes:", weighted_astar_naive.explored)
    print("Naive path length:", weighted_astar_naive.nodes[weighted_goal].g)

Path found: [(0, 0), (1, 0), (2, 0), (3, 0), (3, 1), (3, 2), (4, 2), (4, 3), (4, 4)]
Weighted nodes expanded: 11
Weighted expand processed nodes: [(0, 0), (1, 0), (0, 1), (2, 0), (0, 2), (3, 0), (3, 1), (3, 2), (4, 2), (4, 3), (4, 4)]
Path length: 7
Naive path found: [(0, 0), (1, 0), (2, 0), (3, 0), (3, 1), (3, 2), (4, 2), (4, 3), (4, 4)]
Naive nodes expanded: 15
Naive expand processed nodes: [(0, 0), (1, 0), (0, 1), (2, 0), (0, 2), (3, 0), (3, 1), (1, 2), (3, 2), (2, 2), (1, 3), (4, 2), (1, 4), (4, 3), (4, 4)]
Naive path length: 7


In [19]:
import json

class TreeNode:
    def __init__(self, value=None, children=None):
        self.value = value
        self.children = children if children is not None else []


def build_tree(data):
    if "value" in data:
        return TreeNode(value=data["value"])

    children = [
        build_tree(child)
        for child in data["children"]
    ]

    return TreeNode(children=children)


def alpha_beta(node, alpha=float("-inf"), beta=float("inf"),
               is_maximizing=True, *, stats=None):
    if stats is not None:
        stats["nodes"] += 1

    if not node.children:
        if stats is not None:
            stats["leaves"] += 1
        return node.value

    if is_maximizing:
        best = float("-inf")

        for child in node.children:
            score = alpha_beta(child, alpha, beta, False, stats=stats)
            best = max(best, score)
            alpha = max(alpha, best)

            if alpha >= beta:
                break

        return best

    else:
        best = float("inf")

        for child in node.children:
            score = alpha_beta(child, alpha, beta, True, stats=stats)
            best = min(best, score)
            beta = min(beta, best)

            if alpha >= beta:
                break

        return best


if __name__ == "__main__":
        data = {
        "tree_structure": {
            "is_maximizing_player": True,
            "children": [
                {
                    "is_maximizing_player": False,
                    "children": [
                        {
                            "is_maximizing_player": True,
                            "children": [
                                {"value": 3},
                                {"value": 5}
                            ]
                        },
                        {
                            "is_maximizing_player": True,
                            "children": [
                                {"value": 6},
                                {"value": 9}
                            ]
                        }
                    ]
                },
                {
                    "is_maximizing_player": False,
                    "children": [
                        {
                            "is_maximizing_player": True,
                            "children": [
                                {"value": 6},
                                {"value": 9}
                            ]
                        },
                        {
                            "is_maximizing_player": True,
                            "children": [
                                {"value": 1},
                                {"value": 2}
                            ]
                        }
                    ]
                }
            ]
        }
    }

        tree = build_tree(data["tree_structure"])
        result = alpha_beta(tree)

        print(result)
        tree = build_tree(data["tree_structure"])

        # 根节点为 MAX
        result = alpha_beta(tree)

        print(result)
        


5
5


In [20]:
#比较 Minimax 与不同访问顺序的 Alpha–Beta
def minimax(node, stats, is_maximizing=True):
    stats["nodes"] += 1
    if not node.children:
        stats["leaves"] += 1
        return node.value

    choose = max if is_maximizing else min
    return choose(minimax(child, stats, not is_maximizing) for child in node.children)


def reverse_tree_order(node):
    node.children.reverse()
    for child in node.children:
        reverse_tree_order(child)


tree_data = data["tree_structure"]
original_tree = build_tree(tree_data)

reversed_tree = build_tree(tree_data)
reverse_tree_order(reversed_tree)

right_swapped_tree = build_tree(tree_data)
right_swapped_tree.children[1].children.reverse()

print("算法与访问顺序 | 访问节点 | 求值叶节点 | 根值")
stats = {"nodes": 0, "leaves": 0}
root_value = minimax(original_tree, stats)
print(f"Minimax，从左到右 | {stats['nodes']} | {stats['leaves']} | {root_value}")

for label, trial_tree in [
    ("从左到右", original_tree),
    ("各节点均反序", reversed_tree),
    ("仅交换右侧 MIN 的两个孩子", right_swapped_tree),
]:
    stats = {"nodes": 0, "leaves": 0}
    root_value = alpha_beta(trial_tree, stats=stats)
    print(f"Alpha–Beta，{label} | {stats['nodes']} | {stats['leaves']} | {root_value}")

算法与访问顺序 | 访问节点 | 求值叶节点 | 根值
Minimax，从左到右 | 15 | 8 | 5
Alpha–Beta，从左到右 | 14 | 7 | 5
Alpha–Beta，各节点均反序 | 14 | 7 | 5
Alpha–Beta，仅交换右侧 MIN 的两个孩子 | 11 | 5 | 5


In [21]:
from itertools import combinations, product
grid_p4 = [
    ["A", 1, 1, "X", 2, 1, "T1"],
    [1, "X", 6, "X", 1, "X", 1],
    [1, 1, "T2", 1, 1, 1, 1],
    ["X", 1, "X", 8, "X", 1, "X"],
    [1, 1, 1, 1, "T3", 1, 1],
    [1, "X", 9, "X", 1, "X", 1],
    ["T4", 1, 1, "X", 1, 1, "B"]
]
position = {
    'A': (0, 0),
    'B': (6, 6),
    'T1': (0, 6),
    'T2': (2, 2),
    'T3': (4, 4),
    'T4': (6, 0)
}
def heuristics(current,goal):
    return abs(current[0] - goal[0]) + abs(current[1] - goal[1])
pairs = list(combinations(position.keys(),2)) 
result = {}
for (start,goal) in pairs:
    astar_p4  = A_star(heuristics, position[start], position[goal], grid_p4)
   
    if astar_p4.search():
        print("Path found:", astar_p4.result)
        print("Nodes expanded:", astar_p4.counter)
        print("Expand processed nodes:", astar_p4.explored)
        print("Path length:", astar_p4.nodes[position[goal]].g)
        result[(start,goal)] = astar_p4.nodes[position[goal]].g
    else:
        print("No path found")
print("All results:", result)

Path found: [(0, 0), (1, 0), (2, 0), (2, 1), (3, 1), (4, 1), (4, 2), (4, 3), (4, 4), (5, 4), (6, 4), (6, 5), (6, 6)]
Nodes expanded: 24
Expand processed nodes: [(0, 0), (1, 0), (0, 1), (2, 0), (0, 2), (2, 1), (3, 1), (2, 2), (4, 1), (2, 3), (4, 2), (2, 4), (4, 3), (2, 5), (4, 4), (3, 5), (2, 6), (5, 4), (4, 5), (6, 4), (4, 6), (6, 5), (5, 6), (6, 6)]
Path length: 12
Path found: [(0, 0), (1, 0), (2, 0), (2, 1), (2, 2), (2, 3), (2, 4), (2, 5), (2, 6), (1, 6), (0, 6)]
Nodes expanded: 14
Expand processed nodes: [(0, 0), (0, 1), (0, 2), (1, 0), (2, 0), (2, 1), (2, 2), (2, 3), (2, 4), (1, 4), (2, 5), (2, 6), (1, 6), (0, 6)]
Path length: 10
Path found: [(0, 0), (1, 0), (2, 0), (2, 1), (2, 2)]
Nodes expanded: 7
Expand processed nodes: [(0, 0), (1, 0), (0, 1), (2, 0), (0, 2), (2, 1), (2, 2)]
Path length: 4
Path found: [(0, 0), (1, 0), (2, 0), (2, 1), (3, 1), (4, 1), (4, 2), (4, 3), (4, 4)]
Nodes expanded: 14
Expand processed nodes: [(0, 0), (1, 0), (0, 1), (2, 0), (0, 2), (2, 1), (3, 1), (2, 2)

In [22]:
names = list(position.keys())

matrix = [
    [
        0 if a == b else result.get((a, b), result.get((b, a)))
        for b in names
    ]
    for a in names
]
print("     " + "".join(f"{name:>5}" for name in names))

for name, row in zip(names, matrix):
    print(f"{name:>5}" + "".join(f"{value:>5}" for value in row))

         A    B   T1   T2   T3   T4
    A    0   12   10    4    8    8
    B   12    0    8    8    4   10
   T1   10    8    0    6    6   12
   T2    4    8    6    0    6    6
   T3    8    4    6    6    0    6
   T4    8   10   12    6    6    0


In [23]:
for (a, b), d in list(result.items()):
    result[(b, a)] = d

for name in position:
    result[(name, name)] = 0

values = {
    'T1': 9,
    'T2': 6,
    'T3': 7,
    'T4': 7
}

def build_game_tree(pos_a, pos_b, remaining,
                    score_a, score_b, turn, depth, *, prefer_immediate_gain=False):

    if depth == 0 or not remaining:
        return TreeNode(value=score_a - score_b)

    targets = remaining
    if prefer_immediate_gain:
        current_position = pos_a if turn == 'A' else pos_b
        # 同收益时，稳定排序保留宝藏原有的编号顺序。
        targets = sorted(
            remaining,
            key=lambda target: values[target] - result[(current_position, target)],
            reverse=True,
        )

    children = []
    for target in targets:
        new_remaining = tuple(
            t for t in remaining if t != target
        )

        if turn == 'A':
            gain = values[target] - result[(pos_a, target)]

            child = build_game_tree(
                target, pos_b, new_remaining,
                score_a + gain, score_b,
                'B', depth - 1,
                prefer_immediate_gain=prefer_immediate_gain
            )

        else:
            gain = values[target] - result[(pos_b, target)]

            child = build_game_tree(
                pos_a, target, new_remaining,
                score_a, score_b + gain,
                'A', depth - 1,
                prefer_immediate_gain=prefer_immediate_gain
            )

        children.append(child)

    return TreeNode(children=children)

def solve(H, *, use_alpha_beta=True):
    treasures = tuple(values.keys())

    best_target = None
    best_value = float('-inf')

    stats = {"nodes": 0, "leaves": 0}

    for target in treasures:

        # A 第一次选择宝藏
        gain = values[target] - result[('A', target)]

        remaining = tuple(
            t for t in treasures if t != target
        )

        # 轮到 B
        child = build_game_tree(
            target, 'B', remaining,
            gain, 0, 'B', H - 1
        )


        if use_alpha_beta:
            score = alpha_beta(child, alpha=best_value, is_maximizing=False)
        else:
            score = minimax(child, stats, is_maximizing=False)

        print(f"{target}: {score}")

        if score > best_value:
            best_value = score
            best_target = target


    return {
        "best_target": best_target,
        "best_value": best_value
    }

print("H = 2")
print(solve(2))

print("\nH = 4")
print(solve(4))

print("\nMinimax H = 2")
print(solve(2, use_alpha_beta=False))

print("\nMinimax H = 4")
print(solve(4, use_alpha_beta=False))


H = 2
T1: -4
T2: -1
T3: -2
T4: -2
{'best_target': 'T2', 'best_value': -1}

H = 4
T1: -5
T2: 1
T3: 3
T4: -1
{'best_target': 'T3', 'best_value': 3}

Minimax H = 2
T1: -4
T2: -1
T3: -2
T4: -4
{'best_target': 'T2', 'best_value': -1}

Minimax H = 4
T1: -5
T2: 1
T3: 3
T4: -7
{'best_target': 'T3', 'best_value': 3}


In [24]:
# 题目四：比较不同访问顺序的剪枝效果
def compare_game_search(depth):
    args = ("A", "B", tuple(sorted(values)), 0, 0, "A", depth)
    ascending_tree = build_game_tree(*args)
    descending_tree = build_game_tree(*args)
    reverse_tree_order(descending_tree)
    gain_ordered_tree = build_game_tree(*args, prefer_immediate_gain=True)

    trials = [
        ("Minimax，编号升序", minimax, ascending_tree),
        ("Alpha–Beta，编号升序", alpha_beta, ascending_tree),
        ("Alpha–Beta，编号降序", alpha_beta, descending_tree),
        ("Alpha–Beta，即时净收益降序", alpha_beta, gain_ordered_tree),
    ]
    for label, evaluate, trial_tree in trials:
        stats = {"nodes": 0, "leaves": 0}
        # 从完整的 MAX 根节点开始计数，包含根和实际求值的叶节点。
        value = evaluate(trial_tree, stats=stats)
        print(f"{depth} | {label} | {stats['nodes']} | {stats['leaves']} | {value}")


print("深度 | 算法与访问顺序 | 访问节点 | 求值叶节点 | 根值")
for depth in (2, 4):
    compare_game_search(depth)


深度 | 算法与访问顺序 | 访问节点 | 求值叶节点 | 根值
2 | Minimax，编号升序 | 17 | 12 | -1
2 | Alpha–Beta，编号升序 | 13 | 8 | -1
2 | Alpha–Beta，编号降序 | 16 | 11 | -1
2 | Alpha–Beta，即时净收益降序 | 11 | 6 | -1
4 | Minimax，编号升序 | 65 | 24 | 3
4 | Alpha–Beta，编号升序 | 47 | 16 | 3
4 | Alpha–Beta，编号降序 | 46 | 16 | 3
4 | Alpha–Beta，即时净收益降序 | 37 | 12 | 3
